# hmm-studio — 30-second quickstart

Fit a constrained Hidden Markov Model from 4 lines of Python.

Every object below renders as a rich HTML view in Jupyter — heatmaps, statistics tables, color-coded sequences.

**Install** : `pip install hmm-studio`


## 1. Generate some synthetic time-series data

We simulate 3 latent regimes with well-separated Gaussian observations.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
X = np.concatenate([
    rng.normal(0.0, 1.0, (60, 1)),    # regime A
    rng.normal(5.0, 1.0, (60, 1)),    # regime B
    rng.normal(-3.0, 1.0, (60, 1)),   # regime C
])
X.shape

## 2. Declare a topology

`Topology` describes the structure of the model : how many states, what transitions are allowed, what kind of emissions.

In [ ]:
from hmm_core.topology import Topology, EmissionSpec, FitSpec, InitSpec

topo = Topology(
    name="quickstart_3state",
    n_states=3,
    state_names=["low", "mid", "high"],
    emission=EmissionSpec(type="gaussian", covariance_type="diag", n_features=1),
    allowed_transitions=None,            # ergodic — every transition allowed
    startprob="uniform",
    init=InitSpec(strategy="kmeans", seed=42),
    fit=FitSpec(algorithm="baum_welch", n_iter=100, tol=1e-4),
)
topo                                       # rich HTML view inline below

## 3. Fit

Constrained Baum-Welch on the data. The result is a `FittedModel` with log-likelihood, BIC/AIC, convergence info, and the fitted transition matrix.

In [ ]:
from hmm_core.fit import fit

result = fit(topo, X, seed=42)
result

## 4. Decode

Viterbi gives the most likely state sequence.

In [ ]:
viterbi_states = result.model.predict(X)
print("First 30 states:", viterbi_states[:30])

## Try a left-right constrained topology

Force progression : state must go `low → mid → high`, no back-transitions.

In [ ]:
left_right = Topology(
    name="quickstart_left_right",
    n_states=3,
    state_names=["low", "mid", "high"],
    emission=EmissionSpec(type="gaussian", covariance_type="diag", n_features=1),
    allowed_transitions=[
        ("low", "low"), ("low", "mid"),
        ("mid", "mid"), ("mid", "high"),
        ("high", "high"),
    ],
    startprob="first_state",
    init=InitSpec(strategy="kmeans", seed=42),
    fit=FitSpec(algorithm="baum_welch", n_iter=100, tol=1e-4),
)
left_right    # note the forbidden cells marked with × in the mask

## Next steps

- **NHMM** (covariate-dependent transitions) — see `02_nhmm_crypto.ipynb`
- **GMM-NHMM** (multi-modal regimes) — `src/hmm_core/gmm_nhmm.py`
- **Factorial NHMM** (independent dimensions) — `src/hmm_core/factorial_nhmm.py`
- **Data prep recipes** — `from hmm_core.prep import Pipeline`

Full documentation : [docs/roadmap.md](../docs/roadmap.md) for the architecture and strategy.